# Finite State Machines — Moore, Mealy, and a Sequence Detector

An FSM couples a state register with combinational next-state and output logic. In a **Moore** machine the output depends only on the state; in a **Mealy** machine it also depends on the current input. This notebook draws the **state diagram with the current state lit**, the **transition table**, and the **clock-by-clock timing trace** as a string is fed through a sequence detector.

$$s_{k+1} = \delta(s_k, x_k), \qquad y_k = \begin{cases}\lambda(s_k) & \text{Moore}\\ \lambda(s_k, x_k) & \text{Mealy}\end{cases}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':9})

def fsm_diagram(ax, state_names, edges, current, accepting=None, title=''):
    """edges: list of (src, dst, label). states placed on a circle."""
    m=len(state_names); accepting=accepting or set()
    ang=np.linspace(np.pi/2, np.pi/2-2*np.pi, m, endpoint=False)
    R=1.0; xs,ys=R*np.cos(ang),R*np.sin(ang)
    pos={i:(xs[i],ys[i]) for i in range(m)}
    for (s,d,lab) in edges:
        if s==d:
            ax.annotate(lab,(xs[s]*1.35,ys[s]*1.35),fontsize=7,color='#555',ha='center')
            ax.add_patch(FancyArrowPatch((xs[s]*1.12,ys[s]*1.12),(xs[s]*1.18,ys[s]*1.18),
                         connectionstyle='arc3,rad=3.0',arrowstyle='->',mutation_scale=10,color='#999',lw=1))
        else:
            rad=0.25 if (d-s)%m<=m/2 else -0.25
            ax.add_patch(FancyArrowPatch(pos[s],pos[d],connectionstyle=f'arc3,rad={rad}',
                         arrowstyle='->',mutation_scale=12,color='#aaa',lw=1.2,
                         shrinkA=14,shrinkB=14))
            mx,my=(xs[s]+xs[d])/2,(ys[s]+ys[d])/2
            ax.text(mx+rad*my*0.5,my-rad*mx*0.5,lab,fontsize=7,color='#8e44ad',ha='center',weight='bold')
    for i,nm in enumerate(state_names):
        cur=(i==current)
        ax.add_patch(Circle(pos[i],0.26,fc='#c0392b' if cur else '#eef2f7',ec='#34495e',lw=1.6,zorder=2))
        if i in accepting:
            ax.add_patch(Circle(pos[i],0.31,fc='none',ec='#27ae60',lw=1.6,zorder=2))
        ax.text(*pos[i],nm,ha='center',va='center',fontsize=8.5,color='white' if cur else '#333',weight='bold',zorder=3)
    ax.set_xlim(-1.6,1.6); ax.set_ylim(-1.6,1.6); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title,fontsize=9.5)
print('primitives ready')


primitives ready


## Moore vs Mealy — Where the Output Comes From

Both machines recognise the same languages, but a Moore output is a label *inside* each state circle (stable for the whole cycle), while a Mealy output is a label *on the transition arrow* (it can change mid-cycle when the input changes). The block diagrams below highlight the differing output paths.


In [2]:
def draw_moore_mealy(kind):
    fig,ax=plt.subplots(figsize=(8,3.2)); ax.set_xlim(0,11); ax.set_ylim(0,4); ax.axis('off')
    # state register
    ax.add_patch(Rectangle((4,1.2),1.8,1.6,fc='#eef2f7',ec='#34495e',lw=1.6)); ax.text(4.9,2.0,'state\nreg',ha='center',va='center',weight='bold',fontsize=8)
    ax.add_patch(Rectangle((1.2,1.4),1.6,1.2,fc='#eef2f7',ec='#34495e',lw=1.4)); ax.text(2.0,2.0,'next-\nstate',ha='center',va='center',fontsize=7.5)
    ax.annotate('',xy=(4,2.0),xytext=(2.8,2.0),arrowprops=dict(arrowstyle='->',color='#34495e',lw=1.4))
    ax.annotate('x',xy=(0.6,2.6),fontsize=9,color='#2471a3',weight='bold')
    ax.annotate('',xy=(1.2,2.3),xytext=(0.4,2.6),arrowprops=dict(arrowstyle='->',color='#2471a3',lw=1.3))
    # feedback state
    ax.annotate('',xy=(2.0,1.4),xytext=(5.0,1.2),arrowprops=dict(arrowstyle='->',color='#888',lw=1.2,connectionstyle='arc3,rad=0.4'))
    # output logic
    ax.add_patch(Rectangle((7.4,1.4),1.6,1.2,fc='#fdf0e6',ec='#e67e22',lw=1.6)); ax.text(8.2,2.0,'output\nlogic',ha='center',va='center',fontsize=7.5)
    ax.annotate('',xy=(7.4,2.0),xytext=(5.8,2.0),arrowprops=dict(arrowstyle='->',color='#34495e',lw=1.4))
    ax.annotate('y',xy=(9.6,2.0),fontsize=9,color='#c0392b',weight='bold')
    ax.annotate('',xy=(9.5,2.0),xytext=(9.0,2.0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=1.4))
    if kind=='Mealy':
        # extra input path into output logic
        ax.annotate('',xy=(8.2,1.4),xytext=(0.6,0.5),arrowprops=dict(arrowstyle='->',color='#2471a3',lw=1.3,ls='--',connectionstyle='arc3,rad=-0.2'))
        ax.text(4.5,0.35,'input also feeds output (Mealy)',fontsize=8,color='#2471a3')
    else:
        ax.text(8.2,1.0,'output from state only (Moore)',ha='center',fontsize=8,color='#e67e22')
    ax.set_title(f'{kind} machine datapath',fontsize=10)
    plt.tight_layout(); plt.show()
w_kind=widgets.ToggleButtons(options=['Moore','Mealy'],value='Moore',description='type:')
display(w_kind,widgets.interactive_output(draw_moore_mealy,{'kind':w_kind}))


ToggleButtons(description='type:', options=('Moore', 'Mealy'), value='Moore')

Output()

## Sequence Detector '101' (Moore) — Diagram and Live Run

This Moore FSM raises its output when the bit pattern $101$ has just been seen (overlapping allowed). States track *how much of the pattern matches so far*: $S_0$ (nothing), $S_1$ (saw 1), $S_2$ (saw 10), $S_3$ (saw 101, output=1). Feed a bit string and step through it to watch the current state move around the diagram.


In [3]:
# Moore '101' detector transition function: state x input -> next state
NEXT = {
  (0,0):0,(0,1):1,
  (1,0):2,(1,1):1,
  (2,0):0,(2,1):3,
  (3,0):2,(3,1):1}
NAMES=['S0','S1','S2','S3']
EDGES=[(0,0,'0'),(0,1,'1'),(1,2,'0'),(1,1,'1'),(2,0,'0'),(2,3,'1'),(3,2,'0'),(3,1,'1')]
OUT_MOORE={0:0,1:0,2:0,3:1}

def run_moore(bits, step):
    s=0; trace=[(0, None, OUT_MOORE[0])]
    for x in bits:
        s2=NEXT[(s,x)]; trace.append((s2,x,OUT_MOORE[s2])); s=s2
    step=min(step,len(bits))
    cur_state=trace[step][0]
    fig=plt.figure(figsize=(9,4))
    ax1=fig.add_axes([0.02,0.05,0.5,0.9])
    fsm_diagram(ax1,NAMES,EDGES,cur_state,accepting={3},title="Moore '101' detector")
    ax2=fig.add_axes([0.58,0.15,0.4,0.7]); ax2.axis('off')
    ax2.set_xlim(0,len(bits)+1); ax2.set_ylim(0,3)
    ax2.text(0,2.6,'input: ',fontsize=10)
    for i,x in enumerate(bits):
        c='#8e44ad' if i==step-1 and step>0 else '#333'
        ax2.text(1.0+i*0.7,2.6,str(x),fontsize=12,color=c,weight='bold' if i==step-1 else 'normal',ha='center')
    outs=[trace[k][2] for k in range(1,step+1)]
    ax2.text(0,1.6,'output:',fontsize=10)
    for i,o in enumerate(outs):
        ax2.text(1.0+i*0.7,1.6,str(o),fontsize=12,color='#c0392b' if o else '#888',weight='bold' if o else 'normal',ha='center')
    ax2.text(0,0.6,f'step {step}/{len(bits)}  state={NAMES[cur_state]}  out={trace[step][2]}',fontsize=9,color='#555')
    if step>0 and trace[step][2]==1:
        ax2.text(0,0.1,'pattern 101 detected!',fontsize=10,color='#27ae60',weight='bold')
    plt.show()

def _wrap(bits_str, step):
    bits=[int(c) for c in bits_str if c in '01']
    run_moore(bits, step)
w_bits=widgets.Text(value='1101011010',description='input bits:',layout=widgets.Layout(width='400px'))
w_step=widgets.IntSlider(value=0,min=0,max=12,description='step:')
display(widgets.VBox([w_bits,w_step]),widgets.interactive_output(_wrap,{'bits_str':w_bits,'step':w_step}))


Output()

## Transition Table — The FSM as Data

The same machine written as a table: for each (state, input) it lists the next state and the Moore output. This is exactly what gets synthesised into next-state combinational logic plus the state register.


In [4]:
def show_table():
    print(f"{'state':>6} {'in':>3} | {'next':>5} {'out':>4}")
    print('-'*26)
    for s in range(4):
        for x in (0,1):
            ns=NEXT[(s,x)]
            print(f'{NAMES[s]:>6} {x:>3} | {NAMES[ns]:>5} {OUT_MOORE[ns]:>4}')
        if s<3: print()
show_table()


 state  in |  next  out
--------------------------
    S0   0 |    S0    0
    S0   1 |    S1    0

    S1   0 |    S2    0
    S1   1 |    S1    0

    S2   0 |    S0    0
    S2   1 |    S3    1

    S3   0 |    S2    0
    S3   1 |    S1    0


## Moore vs Mealy on the Same Input — Timing Difference

A Mealy '101' detector needs one fewer state and asserts its output *during* the cycle the final 1 arrives, whereas the Moore output appears the cycle after, when the machine has entered the accepting state. The traces below overlay both outputs for the same input stream.


In [5]:
# Mealy '101': states M0(seen nothing/0), M1(seen 1), M2(seen 10); output on transition
MNEXT={(0,0):0,(0,1):1,(1,0):2,(1,1):1,(2,0):0,(2,1):1}
# Mealy output: 1 exactly on M2 --(input 1)--> M1
def mealy_out(s,x): return 1 if (s==2 and x==1) else 0

def compare_timing(bits_str):
    bits=[int(c) for c in bits_str if c in '01']
    n=len(bits)
    # Moore
    s=0; moore=[]
    for x in bits:
        s=NEXT[(s,x)]; moore.append(OUT_MOORE[s])
    # Mealy
    ms=0; mealy=[]
    for x in bits:
        mealy.append(mealy_out(ms,x)); ms=MNEXT[(ms,x)]
    t=np.arange(n)
    fig,axes=plt.subplots(3,1,figsize=(8.5,3.4),sharex=True)
    axes[0].step(t,bits,where='mid',color='#2471a3',lw=2); axes[0].set_ylabel('input',rotation=0,ha='right')
    axes[1].step(t,mealy,where='mid',color='#27ae60',lw=2); axes[1].set_ylabel('Mealy y',rotation=0,ha='right')
    axes[2].step(t,moore,where='mid',color='#c0392b',lw=2); axes[2].set_ylabel('Moore y',rotation=0,ha='right')
    for ax in axes:
        ax.set_ylim(-0.3,1.3); ax.set_yticks([0,1]); ax.grid(True,alpha=0.3)
        ax.set_xticks(t); ax.set_xticklabels(bits)
    axes[-1].set_xlabel('input bit per clock')
    fig.suptitle('Mealy asserts one cycle earlier than Moore',fontsize=9.5)
    plt.tight_layout(); plt.show()
w_cmp=widgets.Text(value='1101011010',description='input bits:',layout=widgets.Layout(width='400px'))
display(w_cmp,widgets.interactive_output(compare_timing,{'bits_str':w_cmp}))


Text(value='1101011010', description='input bits:', layout=Layout(width='400px'))

Output()

## A Practical Moore FSM — Traffic Light Controller

State machines shine in control. This Moore machine cycles Green -> Yellow -> Red with a dwell counter, the output (the lit lamp) being a pure function of state. Step the clock and watch the active state and the corresponding lamp.


In [6]:
LIGHTS=['GREEN','YELLOW','RED']
DWELL=[3,1,3]
COLORS=['#27ae60','#f1c40f','#c0392b']
def traffic(step):
    seq=[]
    for i,d in enumerate(DWELL):
        seq+= [i]*d
    period=len(seq)
    cur=seq[step % period]
    fig=plt.figure(figsize=(8.5,3.4))
    ax1=fig.add_axes([0.02,0.05,0.5,0.9])
    edges=[(0,1,'t>=Tg'),(1,2,'t>=Ty'),(2,0,'t>=Tr')]
    fsm_diagram(ax1,LIGHTS,edges,cur,title='traffic Moore FSM')
    ax2=fig.add_axes([0.62,0.1,0.25,0.8]); ax2.axis('off'); ax2.set_xlim(0,2); ax2.set_ylim(0,4)
    for i,(nm,c) in enumerate(zip(LIGHTS,COLORS)):
        lit=(i==cur)
        ax2.add_patch(Circle((1,3-i),0.35,fc=c if lit else '#eee',ec='#333',lw=1.5))
        ax2.text(1.6,3-i,nm,fontsize=9,va='center',color=c if lit else '#aaa',weight='bold' if lit else 'normal')
    ax2.set_title(f'step {step}: {LIGHTS[cur]}',fontsize=9)
    plt.show()
w_ts=widgets.IntSlider(value=0,min=0,max=20,description='clock:')
display(w_ts,widgets.interactive_output(traffic,{'step':w_ts}))


IntSlider(value=0, description='clock:', max=20)

Output()